# 08. 🧹 리소스 정리

> **워크샵이 끝나면 반드시 이 노트북을 실행하세요.** 실시간 엔드포인트는 삭제 전까지 시간당 과금됩니다.

이 노트북은 워크샵에서 생성한 모든 SageMaker 리소스와 S3 데이터를 삭제합니다.

## 0. 환경 복원

In [ ]:
import boto3
import sagemaker

sess = sagemaker.Session()
%store -r bucket prefix endpoint_name region

sm = boto3.client("sagemaker", region_name=region)
s3 = boto3.resource("s3", region_name=region)
print("region  :", region)
print("bucket  :", bucket)
print("prefix  :", prefix)
print("endpoint:", endpoint_name)

## 1. 엔드포인트 삭제

Endpoint → Endpoint Configuration → Model 순으로 삭제합니다.

In [ ]:
# 엔드포인트 삭제
try:
    sm.delete_endpoint(EndpointName=endpoint_name)
    print("deleted endpoint:", endpoint_name)
except Exception as e:
    print("endpoint:", e)

# 엔드포인트 구성 삭제
try:
    sm.delete_endpoint_config(EndpointConfigName=endpoint_name)
    print("deleted endpoint config:", endpoint_name)
except Exception as e:
    print("endpoint config:", e)

## 2. 모델 삭제

배포용 모델뿐 아니라, HPO 튜닝에서 생성된 모든 모델을 삭제합니다.

In [ ]:
# 워크샵에서 생성한 모든 모델 삭제 (student-xgb, student-xgb-hpo 등)
search_prefixes = ["student-xgb", "student-success"]
deleted_models = 0
for name_prefix in search_prefixes:
    paginator = sm.get_paginator("list_models")
    for page in paginator.paginate(NameContains=name_prefix):
        for m in page["Models"]:
            sm.delete_model(ModelName=m["ModelName"])
            deleted_models += 1
            print("  deleted model:", m["ModelName"])
print(f"total models deleted: {deleted_models}")

## 3. 학습 작업 / 튜닝 작업 정리

학습·튜닝 작업 자체는 삭제 API가 없지만(기록은 자동 보관), **남은 Endpoint Config**가 있으면 삭제합니다.

In [ ]:
# 남은 엔드포인트 구성 삭제
paginator = sm.get_paginator("list_endpoint_configs")
deleted_configs = 0
for page in paginator.paginate(NameContains="student"):
    for cfg in page["EndpointConfigs"]:
        sm.delete_endpoint_config(EndpointConfigName=cfg["EndpointConfigName"])
        deleted_configs += 1
        print("  deleted config:", cfg["EndpointConfigName"])
print(f"total endpoint configs deleted: {deleted_configs}")

## 4. S3 워크샵 데이터 삭제

워크샵에서 사용한 S3 prefix 하위의 모든 객체를 삭제합니다.

In [ ]:
bucket_obj = s3.Bucket(bucket)
objects = list(bucket_obj.objects.filter(Prefix=f"{prefix}/"))
if objects:
    bucket_obj.delete_objects(Delete={"Objects": [{"Key": o.key} for o in objects]})
    print(f"deleted {len(objects)} objects from s3://{bucket}/{prefix}/")
else:
    print("no objects to delete")

## 5. (선택) S3 버킷 삭제

버킷에 다른 데이터가 없다면 버킷 자체를 삭제할 수 있습니다. 다른 용도로 사용 중이라면 **건너뛰세요**.

In [ ]:
# 주의: 버킷의 모든 데이터가 삭제됩니다. 필요한 경우에만 주석을 해제하세요.
# remaining = list(bucket_obj.objects.limit(1))
# if not remaining:
#     bucket_obj.delete()
#     print("deleted bucket:", bucket)
# else:
#     print("bucket not empty, skipping deletion")
print("(bucket deletion skipped - uncomment above if needed)")

✅ **정리 완료!**

남은 리소스가 없는지 SageMaker 콘솔에서 확인하세요:
- **Inference > Endpoints** — 활성 엔드포인트 없는지
- **Inference > Models** — 등록된 모델 없는지
- **S3 콘솔** — 워크샵 버킷/prefix 삭제 확인

수고하셨습니다! 🎉